In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

In [3]:
# --- 1. Load the dataset ---
df = pd.read_csv('/kaggle/input/crypto-qna/qna-clean.csv')

print("DataFrame head:")
print(df.head())

print("\nDataFrame info:")
df.info()

DataFrame head:
   Unnamed: 0.2  Unnamed: 0.1      id  comment_score  Unnamed: 0  \
0         20804           NaN  t6iulw              1       20422   
1         27114           NaN  p42oib              1       28026   
2         10175           NaN  n8fd55              2        7288   
3         19865           NaN  q9b2eq              1       19297   
4         23937           NaN  qupurq              2       24255   

        subreddit  created_utc  \
0  cryptocurrency   1646400087   
1  cryptocurrency   1628921432   
2  cryptocurrency   1620570659   
3  cryptocurrency   1634387481   
4  cryptocurrency   1637008822   

                                               title  \
0  Are you planning to donate crypto in the next ...   
1  What are some good gaming projects in the Cryp...   
2  When transferring coins between exchanges, use...   
3  which low cap do you think can become your nex...   
4                   Will crypto be huge in 20 years?   

                                 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


In [4]:
# --- 2. Split data into train, validation, and test sets ---
# First, split into train and temp (80% train, 20% temp)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)

# Then, split temp into validation and test (50% validation, 50% test of the temp_df)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"\nTraining set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")


Training set size: 20229
Validation set size: 2529
Test set size: 2529


In [5]:
# --- 3. Define the custom Dataset class ---
class RelevanceDataset(Dataset):
    def __init__(self, topics, comments, labels, tokenizer, max_length):
        self.topics = topics
        self.comments = comments
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        topic = str(self.topics[idx])
        comment = str(self.comments[idx])
        label = int(self.labels[idx])

        # Tokenize the pair
        encoding = self.tokenizer(
            topic,
            comment,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt' # Return PyTorch tensors
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [41]:
# --- 4. Calculate Class Weights ---
# Get all labels from your full dataset to compute accurate class weights
classes = np.array([0, 1])  # Explicitly set class order if you have binary classification
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=all_labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print(f"\nCalculated Class Weights: {class_weights_tensor.tolist()}")
print(f" (Weights are [Weight for Class 0, Weight for Class 1])")


Calculated Class Weights: [0.5860798358917236, 3.4042811393737793]
 (Weights are [Weight for Class 0, Weight for Class 1])


In [42]:
# --- 5. Choose a pre-trained model and tokenizer ---
model_name = "roberta-base" # Based on previous discussions
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

MAX_LENGTH = 512 # Based on previous discussions for better coverage

# Create dataset objects
train_dataset = RelevanceDataset(train_df['MAIN'].tolist(), train_df['comment_body'].tolist(), train_df['relevance'].tolist(), tokenizer, MAX_LENGTH)
val_dataset = RelevanceDataset(val_df['MAIN'].tolist(), val_df['comment_body'].tolist(), val_df['relevance'].tolist(), tokenizer, MAX_LENGTH)
test_dataset = RelevanceDataset(test_df['MAIN'].tolist(), test_df['comment_body'].tolist(), test_df['relevance'].tolist(), tokenizer, MAX_LENGTH)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [43]:
# --- 6. Define compute_metrics function ---
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary') # Use 'binary' for binary classification
    precision = precision_score(labels, predictions, average='binary')
    recall = recall_score(labels, predictions, average='binary')
    return {"accuracy": accuracy, "f1": f1, "precision": precision, "recall": recall}

In [50]:
# --- 7. Custom Trainer to apply Weighted Loss ---
class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):  # <--- Add **kwargs
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Create and move class weights tensor to the model's device
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(next(model.parameters()).device)
        )

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


In [51]:
# --- 8. Training arguments ---
training_args = TrainingArguments(
    output_dir="./roberta-results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)

In [52]:
# --- 9. Initialize the Custom Trainer ---
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor
)

In [55]:
# --- 10. Train the model ---
print("\nStarting model training with weighted loss...")
trainer.train()


Starting model training with weighted loss...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.562500,0.617426,0.758007,0.426966,0.317107,0.653295
2,0.525400,0.579563,0.877817,0.480672,0.581301,0.409742
3,0.491200,0.567483,0.807829,0.467105,0.378330,0.610315
4,0.409300,0.578152,0.782918,0.464390,0.352071,0.681948
5,0.345000,0.666620,0.818505,0.475429,0.395437,0.595989


TrainOutput(global_step=3165, training_loss=0.4663618645208518, metrics={'train_runtime': 5664.6706, 'train_samples_per_second': 17.855, 'train_steps_per_second': 0.559, 'total_flos': 2.694073029147648e+16, 'train_loss': 0.4663618645208518, 'epoch': 5.0})

In [56]:
# --- 11. Evaluate the final (best) model on the test set ---
print("\nEvaluating model on the test set...")

test_results = trainer.evaluate(test_dataset)
print(f"Test set results: {test_results}")


Evaluating model on the test set...


Test set results: {'eval_loss': 0.5591558814048767, 'eval_accuracy': 0.8774219058916568, 'eval_f1': 0.4850498338870432, 'eval_precision': 0.6431718061674009, 'eval_recall': 0.3893333333333333, 'eval_runtime': 44.2277, 'eval_samples_per_second': 57.181, 'eval_steps_per_second': 1.809, 'epoch': 5.0}


In [63]:
from transformers import AutoModelForSequenceClassification

model_path = "/kaggle/working/final_model"

model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    local_files_only=True  # 👈 this prevents it from looking on the internet
)
test_results = trainer.evaluate(test_dataset)
print(f"Test set results: {test_results}")

Test set results: {'eval_loss': 0.5591558814048767, 'eval_accuracy': 0.8774219058916568, 'eval_f1': 0.4850498338870432, 'eval_precision': 0.6431718061674009, 'eval_recall': 0.3893333333333333, 'eval_runtime': 44.4579, 'eval_samples_per_second': 56.885, 'eval_steps_per_second': 1.799, 'epoch': 5.0}


In [57]:
# --- 12. Save the model ---
# Save the model to Kaggle working directory
model.save_pretrained("/kaggle/working/final_model")
tokenizer.save_pretrained("/kaggle/working/final_model")

('/kaggle/working/final_model/tokenizer_config.json',
 '/kaggle/working/final_model/special_tokens_map.json',
 '/kaggle/working/final_model/vocab.json',
 '/kaggle/working/final_model/merges.txt',
 '/kaggle/working/final_model/added_tokens.json',
 '/kaggle/working/final_model/tokenizer.json')

In [65]:
# --- 12. Run the Inference and Save Output CSV---
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# --- Config ---
MODEL_PATH = "/kaggle/working/final_model"  # Replace with your actual checkpoint folder
MODEL_NAME = "roberta-base"
MAX_LENGTH = 512
BATCH_SIZE = 32
TEST_FILE = "/kaggle/input/crypto-qna-test/CRYPTO_QnA_TEST.csv"
OUTPUT_FILE = "crypto_test_qna.csv"

# --- Load tokenizer and model ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# --- Dataset for inference ---
class InferenceDataset(Dataset):
    def __init__(self, topics, comments, tokenizer, max_length):
        self.encodings = tokenizer(
            topics, comments,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )

    def __len__(self):
        return self.encodings["input_ids"].size(0)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx]
        }

# --- Load test data ---
df = pd.read_csv(TEST_FILE)
dataset = InferenceDataset(
    topics=df["MAIN"].astype(str).tolist(),
    comments=df["comment_body"].astype(str).tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

# --- Inference Trainer ---
inference_args = TrainingArguments(
    output_dir="./inference_results",
    per_device_eval_batch_size=BATCH_SIZE,
    report_to="none"
)

trainer = Trainer(model=model, args=inference_args)

# --- Run prediction ---
print("\nRunning inference...")
preds = trainer.predict(dataset)
logits = preds.predictions
pred_labels = np.argmax(logits, axis=1)

# --- Save to CSV ---
df["relevance"] = pred_labels
df_out = df[["title", "selftext", "MAIN", "comment_body", "relevance"]]
df_out.to_csv(OUTPUT_FILE, index=False)

print(f"\n✅ Inference complete. Saved to: {OUTPUT_FILE}")
print(df_out.head())


Running inference...



✅ Inference complete. Saved to: crypto_test_qna.csv
                                               title  \
0  Which exchange to use to see holdings increase...   
1  The end of this year is approaching, how would...   
2                        When do you pull out? (Ha!)   
3  ETH, BTC, ADA, ATOM, ALGO, any other promising...   
4                  Convince me any of this has value   

                                            selftext  \
0  Wazirx doesn't show how much a portfolio has c...   
1  As the end of the year approaches, I am intere...   
2  So I promised my SO that I would only invest a...   
3  These so far are the ones I’ve locked in and c...   
4  I’ve been watching crypto from the outside for...   

                                                MAIN  \
0  Which exchange to use to see holdings increase...   
1  the end of this year is approaching, how would...   
2  when do you pull out? (ha!) so i promised my s...   
3  eth, btc, ada, atom, algo, any other promising

In [66]:
import shutil
import os

# Define the directory where your model checkpoints are saved
# This should be the 'output_dir' you set in your TrainingArguments
checkpoint_directory = "/kaggle/working/final_model"

# Define the desired name for your output zip file
# The .zip extension will be added automatically
output_zip_filename = "qna_relevance_model_checkpoints"

# Check if the checkpoint directory exists
if os.path.exists(checkpoint_directory) and os.path.isdir(checkpoint_directory):
    try:
        print(f"Zipping the checkpoint directory: {checkpoint_directory}...")
        # Create the zip archive
        # base_name: The name of the archive file to create (without extension).
        # format: The archive format ('zip', 'tar', 'gztar', 'bztar', 'xztar').
        # root_dir: The directory that will be archived. Its contents will be placed directly in the zip.
        # The resulting zip file will be created in the current working directory.
        shutil.make_archive(base_name=output_zip_filename, format='zip', root_dir=checkpoint_directory)

        print(f"Successfully created '{output_zip_filename}.zip'")
        print(f"You can now download the file: {output_zip_filename}.zip")
    except Exception as e:
        print(f"An error occurred while zipping the folder: {e}")
else:
    print(f"Error: The checkpoint directory '{checkpoint_directory}' was not found.")
    print("Please ensure your model training completed successfully and saved checkpoints to this directory.")

Zipping the checkpoint directory: /kaggle/working/final_model...
Successfully created 'qna_relevance_model_checkpoints.zip'
You can now download the file: qna_relevance_model_checkpoints.zip
